# Week 11 — Fine-tuned stage (model adaptation)

Run the three Week-10 OBB specialists (haze/jpeg/noise) through the **same** AABB + VOC-AP pipeline as the W8 distorted and W9 restored stages, so the specialist mAP is finally comparable to the clean baseline.

**Axis note:** all curves are plotted against the *distorted* (input) SNR — a shared degradation-severity axis. Distorted and fine-tuned share identical input images, so the vertical gap between them at each point reads directly as the improvement from adapting the model; the restored (green) curve shifts in x because W9 recomputed SNR restored-vs-clean. This compares the two recovery strategies: enhance the image (W9) vs adapt the model (W10).

In [1]:
from pathlib import Path
import pandas as pd
from src.figures import plot_distorted_vs_restored

REPO   = Path.cwd().parent
DSW    = REPO / 'results' / 'distortion_sweep'
RSW    = REPO / 'results' / 'restoration_sweep'
FSW    = REPO / 'results' / 'finetuned_sweep'
CLEAN  = REPO / 'results' / 'clean'
OUTFIG = REPO / 'outputs' / 'figures'
OUTFIG.mkdir(parents=True, exist_ok=True)
XLABEL = 'Distorted SNR (dB) — degradation severity'

def agg(sweep, fname, col):
    df = pd.read_csv(sweep / fname, dtype={'level': str})
    return df.groupby(['distortion', 'level', 'snr_db_mean'], as_index=False)[col].mean()

clean_map = pd.read_csv(CLEAN / 'perclass_detections.csv')['ap_iou50'].mean()
print(f'clean mAP@0.5 = {clean_map:.3f}')

/Users/yairnachum/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


clean mAP@0.5 = 0.732


## mAP@0.5 — distorted (red) vs restored (green) vs fine-tuned (blue)

In [2]:
plot_distorted_vs_restored(
    agg(DSW, 'perclass_detections.csv', 'ap_iou50'),
    agg(RSW, 'perclass_detections.csv', 'ap_iou50'),
    value_col='ap_iou50', title='mAP@0.5: distorted vs restored vs fine-tuned',
    ylabel='mAP@0.5', out_path=OUTFIG / 'finetuned_recovery_map_vs_snr.png',
    clean_baseline=clean_map, xlabel=XLABEL,
    df_third=agg(FSW, 'perclass_detections.csv', 'ap_iou50'), third_label='fine-tuned',
)

## Per-distortion recovery table (mean mAP gain)

In [3]:
def fam_map(sweep):
    return pd.read_csv(sweep / 'perclass_detections.csv', dtype={'level': str}).groupby('distortion')['ap_iou50'].mean()

d, r, f = fam_map(DSW), fam_map(RSW), fam_map(FSW)
pd.DataFrame({
    'distorted mAP':      d.round(3),
    'restored mAP':       r.round(3),
    'fine-tuned mAP':     f.round(3),
    'FT − distorted':     (f - d).round(3),
    'FT − restored':      (f - r).round(3),
})

,distorted mAP,restored mAP,fine-tuned mAP,FT − distorted,FT − restored
distortion,,,,,
haze,0.404,0.666,0.512,0.108,-0.153
jpeg,0.364,0.406,0.434,0.070,0.028
noise,0.304,0.350,0.499,0.194,0.148
